# 00 — Smoke test

Tiny end-to-end run to verify the install + code chain **before**
committing to the full 128³ / 20 sims / 40 epochs demo.

What this does:

- 32³ box in a 64 Mpc side (dx = 2 Mpc kept)
- 2 train + 1 val 21cmFAST coeval cubes at `<x_HI> ≈ 0.5`
- HERA observation via `tuesday.core.observe_coeval`
- 3 training epochs on a small U-Net (base = 8)
- plots of the signal, observed cube, training curves, and a prediction slice

On a laptop CPU this should finish in roughly 5–10 minutes, most of
which is 21cmFAST. Re-running skips existing cubes.

If this notebook completes cleanly the full `ml_pipeline_demo.ipynb`
will too — just bigger and slower.

In [ ]:
import sys, pathlib
HERE = pathlib.Path().resolve()
PROJECT_ROOT = HERE.parents[1]              # .../noisy_reconstruction
PHD_ROOT     = HERE.parents[2]              # .../PhD
sys.path.insert(0, str(PHD_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from code.ml.config   import DemoConfig
from code.ml.simulate import generate_dataset
from code.ml.noise    import observe_Tb
from code.ml.train    import train, load_best

# ---- tiny smoke-test config ------------------------------------------------
cfg = DemoConfig(
    hii_dim       = 32,
    box_len       = 64.0,       # dx = 2 Mpc
    n_train       = 2,
    n_val         = 1,
    seed0         = 90000,      # own seed range so it doesn't clash with the full demo
    epochs        = 3,
    base_channels = 8,
    device        = 'cpu',
    data_dir = PROJECT_ROOT / 'data' / 'cubes_smoketest',
    ckpt_dir = PROJECT_ROOT / 'data' / 'checkpoints_smoketest',
)
cfg

## 1. Environment sanity check

Fail fast with a clear message if any of the four main deps is missing.

In [ ]:
missing = []
for pkg in ('py21cmfast', 'py21cmsense', 'tuesday', 'torch'):
    try:
        __import__(pkg)
    except ImportError as e:
        missing.append((pkg, str(e)))

if missing:
    print('MISSING DEPS — install before continuing:')
    for pkg, err in missing:
        print(f'  - {pkg}: {err}')
    print('\nFrom PhD/noise_analysis:  uv sync && uv add py21cmfast torch')
else:
    print('env OK: py21cmfast, py21cmsense, tuesday, torch all importable')

## 2. Generate the 3 tiny 21cmFAST cubes

In [ ]:
paths = generate_dataset(cfg)
print(f"z used: {paths['z']:.3f}")
print(f"train cubes: {[p.name for p in paths['train']]}")
print(f"val cubes:   {[p.name for p in paths['val']]}")

## 3. Observe one cube through HERA

If `observe_coeval` succeeds and returns something with the same shape
as the input, the tuesday / py21cmsense side is wired up correctly.

In [ ]:
sample = np.load(paths['train'][0])
Tb_obs = observe_Tb(sample['Tb'], float(sample['z']), cfg, seed=0)

print(f"Tb clean    : shape={sample['Tb'].shape}  mean={sample['Tb'].mean():+.2f}  std={sample['Tb'].std():.2f}  mK")
print(f"Tb observed : shape={Tb_obs.shape}       mean={Tb_obs.mean():+.2f}  std={Tb_obs.std():.2f}  mK")
assert Tb_obs.shape == sample['Tb'].shape, 'shape mismatch!'

mid = cfg.hii_dim // 2
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, data, title, cmap in [
    (axes[0], sample['Tb'][:, :, mid], 'Tb clean [mK]',    'viridis'),
    (axes[1], Tb_obs[:, :, mid],       'Tb observed [mK]', 'viridis'),
    (axes[2], sample['vz'][:, :, mid], 'v_z [km/s]',       'RdBu_r'),
]:
    im = ax.imshow(data.T, origin='lower', cmap=cmap)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.85)
plt.show()

## 4. Train for 3 epochs

Goal here is not to learn anything meaningful — just to exercise
`TbToVzDataset → DataLoader → UNet3D.forward → backward → optimizer.step
→ checkpointing` without crashes.

In [ ]:
history = train(cfg, paths['train'], paths['val'])
assert len(history['train_loss']) == cfg.epochs
assert (cfg.ckpt_dir / 'best.pt').exists(), 'best.pt not written'
print('\nsmoke-test complete — the full ml_pipeline_demo.ipynb should now run too.')

## 5. Quick prediction slice

In [ ]:
import torch
model, stats, device = load_best(cfg)

val = np.load(paths['val'][0])
x = (observe_Tb(val['Tb'], float(val['z']), cfg, seed=999) - stats.x_mean) / stats.x_std
x = torch.from_numpy(x).float().unsqueeze(0).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(x).squeeze().cpu().numpy() * stats.y_std + stats.y_mean

mid = cfg.hii_dim // 2
vmax = float(np.percentile(np.abs(val['vz']), 99))
fig, axes = plt.subplots(1, 2, figsize=(7, 3.2), constrained_layout=True)
for ax, data, title in [
    (axes[0], val['vz'][:, :, mid], 'v_z true'),
    (axes[1], pred[:, :, mid],       'v_z pred (3 epochs — not trained!)'),
]:
    im = ax.imshow(data.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, shrink=0.85)
plt.show()